In [81]:
import os

base = "/kaggle/input/datasets/anshuman9468"
print(os.listdir(base))

['duality-gtbit']


In [82]:
print(os.listdir("/kaggle/input/datasets/anshuman9468/duality-gtbit/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"))

['val', 'train']


In [83]:
data_dir = "/kaggle/input/datasets/anshuman9468/duality-gtbit/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train"

print(os.listdir(data_dir))

['Segmentation', 'Color_Images']


In [84]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [85]:
!nvidia-smi

Thu Mar 19 21:01:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             34W /   70W |   13047MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [86]:
# ============================================================================
# 🚀 FAST + HIGH IoU DEEPLABV3 (OPTIMIZED)
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models.segmentation import deeplabv3_resnet50
from PIL import Image
import numpy as np
import os
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_SIZE = 384   # 🔥 reduced → faster
BATCH_SIZE = 6   # 🔥 higher batch → stable + fast

value_map = {
    0: 0, 100: 1, 200: 2, 300: 3, 500: 4,
    550: 5, 700: 6, 800: 7, 7100: 8, 10000: 9
}
n_classes = len(value_map)

# ============================================================================
# DATASET
# ============================================================================

class SegDataset(Dataset):
    def __init__(self, root, transform=None, mask_transform=None):
        self.img_dir = os.path.join(root, "Color_Images")
        self.mask_dir = os.path.join(root, "Segmentation")
        self.files = sorted(os.listdir(self.img_dir))

        self.transform = transform
        self.mask_transform = mask_transform

        print(f"{root}: {len(self.files)} samples")

    def __len__(self):
        return len(self.files)

    def convert_mask(self, mask):
        arr = np.array(mask)
        new_mask = np.zeros_like(arr)

        for k, v in value_map.items():
            new_mask[arr == k] = v

        return Image.fromarray(new_mask.astype(np.uint8))

    def __getitem__(self, idx):
        name = self.files[idx]

        img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, name))

        mask = self.convert_mask(mask)

        if self.transform:
            img = self.transform(img)

        if self.mask_transform:
            mask = self.mask_transform(mask).squeeze().long()

        return img, mask

# ============================================================================
# LOSS (BOOST IoU)
# ============================================================================

class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=probs.shape[1])
        targets_one_hot = targets_one_hot.permute(0,3,1,2).float()

        inter = (probs * targets_one_hot).sum((2,3))
        union = probs.sum((2,3)) + targets_one_hot.sum((2,3))

        dice = (2 * inter + 1e-6) / (union + 1e-6)
        return 1 - dice.mean()

# ============================================================================
# METRIC
# ============================================================================

def compute_iou(pred, target, num_classes):
    pred = pred.view(-1)
    target = target.view(-1)

    ious = []
    for cls in range(num_classes):
        inter = ((pred == cls) & (target == cls)).sum().float()
        union = ((pred == cls) | (target == cls)).sum().float()
        if union > 0:
            ious.append((inter / union).item())

    return np.mean(ious) if ious else 0

# ============================================================================
# TRAIN
# ============================================================================

def main():

    root = "/kaggle/input/datasets/anshuman9468/duality-gtbit/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"
    train_dir = os.path.join(root, "train")
    val_dir = os.path.join(root, "val")

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.2,0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    mask_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=Image.NEAREST),
        transforms.ToTensor()
    ])

    train_ds = SegDataset(train_dir, transform, mask_transform)
    val_ds = SegDataset(val_dir, transform, mask_transform)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=2,       # 🔥 faster loading
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=True
    )

    # ============================================================================
    # MODEL (FASTER)
    # ============================================================================

    model = deeplabv3_resnet50(weights="DEFAULT")
    model.classifier[4] = nn.Conv2d(256, n_classes, 1)
    model = model.to(device)

    # freeze batchnorm
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

    # ============================================================================
    # OPTIM
    # ============================================================================

    ce_loss = nn.CrossEntropyLoss()
    dice_loss = DiceLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    scaler = GradScaler()

    best_iou = 0

    # ============================================================================
    # TRAIN LOOP
    # ============================================================================

    for epoch in range(20):   # 🔥 fewer epochs needed now

        model.train()
        total_loss = 0

        for imgs, masks in tqdm(train_loader):

            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()

            with autocast():
                outputs = model(imgs)['out']
                loss = ce_loss(outputs, masks) + 0.5 * dice_loss(outputs, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        # ============================================================================
        # VALIDATION
        # ============================================================================

        model.eval()
        val_iou = 0

        with torch.no_grad():
            for imgs, masks in val_loader:

                imgs, masks = imgs.to(device), masks.to(device)

                with autocast():
                    outputs = model(imgs)['out']

                preds = outputs.argmax(1)
                val_iou += compute_iou(preds, masks, n_classes)

        val_iou /= len(val_loader)

        print(f"\nEpoch {epoch+1}")
        print(f"Loss: {total_loss:.4f}")
        print(f"Val IoU: {val_iou:.4f}")

        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
            print("✅ Saved Best Model")

    print(f"\n🏆 FINAL BEST IoU: {best_iou:.4f}")


if __name__ == "__main__":
    main()

/kaggle/input/datasets/anshuman9468/duality-gtbit/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train: 2857 samples
/kaggle/input/datasets/anshuman9468/duality-gtbit/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val: 317 samples
Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth


100%|██████████| 161M/161M [00:00<00:00, 185MB/s] 
100%|██████████| 476/476 [02:38<00:00,  3.00it/s]



Epoch 1
Loss: 481.5139
Val IoU: 0.6912
✅ Saved Best Model


100%|██████████| 476/476 [02:29<00:00,  3.18it/s]



Epoch 2
Loss: 233.7777
Val IoU: 0.7348
✅ Saved Best Model


...


Epoch 18
Loss: 214.2301
Val IoU: 0.8207


100%|██████████| 476/476 [02:25<00:00,  3.26it/s]



Epoch 19
Loss: 214.2229
Val IoU: 1.0000
✅ Saved Best Model


100%|██████████| 476/476 [02:25<00:00,  3.26it/s]



Epoch 20
Loss: 214.2174
Val IoU: 0.8490

🏆 FINAL BEST IoU: 1.0000


In [87]:
!ls /kaggle/input/datasets/anshuman9468/duality-gtbit


Offroad_Segmentation_testImages  Offroad_Segmentation_Training_Dataset
